In [ ]:
import glob
import os
import statistics
import numpy as np

In [ ]:
from PIL import Image

In [ ]:
split = "train"
label = "good"
folder_name = "Brats"

In [ ]:
if split == "train":
    image_paths = glob.glob(os.path.join(f"../dataset/{folder_name}/train/{label}/", "*.png"))
    segmentation_mask_paths = []
else:
    image_paths = glob.glob(os.path.join(f"../dataset/{folder_name}/{split}/{label}/img/", "*.png"))
    segmentation_mask_paths = glob.glob(os.path.join(f"../dataset/{folder_name}/{split}/{label}/label", "*.png"))

In [ ]:
print("# of images: ", len(image_paths))

In [ ]:
image_path_index = 5
image_path = image_paths[image_path_index]
image = Image.open(image_path)

if split != "train":
    segmentation_mask_path = image_path.replace("img", "label")
    segmentation_mask = Image.open(segmentation_mask_path)

In [ ]:
image

In [ ]:
if split != "train":
    display(segmentation_mask)

In [ ]:
from collections import Counter
Counter(np.asarray(segmentation_mask).flatten().tolist())

In [ ]:
image.size

In [ ]:
image.mode

In [ ]:
# Image sizes statistics

image_sizes = {}
for image_path in image_paths:
    image = Image.open(image_path)
    image_sizes[image.size] = image_sizes.get(image.size, 0) + 1

print(image_sizes)

In [ ]:
# Compute the pixel-value statistics for the input images
image_pixel_means = []

for image_path in image_paths:
    image = Image.open(image_path).convert('L')
    image_arr = np.asarray(image)
    image_pixel_means.append(image_arr.mean())

print("Mean - pixel-level mean of the input images: ", statistics.mean(image_pixel_means))
print("Median - pixel-level mean of the input images: ", statistics.median(image_pixel_means))
print("Stdev - pixel-level mean of the input images: ", statistics.stdev(image_pixel_means))
print("Min - pixel-level mean of the input images: ", min(image_pixel_means))
print("Max - pixel-level mean of the input images: ", max(image_pixel_means))
print("Mode - pixel-level mean of the input images: ", statistics.mode(image_pixel_means))

In [ ]:
if split != "train":
    # Compute the pixel-value statistics for the segmentation masks
    mask_pixel_means = []
    
    for segmentation_mask_path in segmentation_mask_paths:
        segmentation_mask = Image.open(segmentation_mask_path).convert('L')
        segmentation_mask_arr = np.asarray(segmentation_mask)
        mask_pixel_means.append(segmentation_mask_arr.mean())
    
    print("Mean - pixel-level mean of the segmentation masks: ", statistics.mean(mask_pixel_means))
    print("Median - pixel-level mean of the segmentation masks: ", statistics.median(mask_pixel_means))
    print("Stdev - pixel-level mean of the segmentation masks: ", statistics.stdev(mask_pixel_means))
    print("Min - pixel-level mean of the segmentation masks: ", min(mask_pixel_means))
    print("Max - pixel-level mean of the segmentation masks: ", max(mask_pixel_means))
    print("Mode - pixel-level mean of the segmentation masks: ", statistics.mode(mask_pixel_means))

In [ ]:
# Brain MRI dataset

# Train set
# 7500 NORMAL images with 1 channel (grayscale) of size (240, 240)
# All files are .png images

# Validation set

# Test set

### Per-channel statistics - for image normalization

In [3]:
import torch
import tqdm
from torchvision import transforms
from src.utils.utils_evaluation import * 

subpath_list = os.getcwd().split(os.sep)
root_directory_path = os.path.join(*subpath_list[:-1])
current_dir = subpath_list[-1]

TRAIN_DATASET_PATH = f"/{root_directory_path}/BMAD/dataset/BraTS2021_slice/train"
print("Dataset path: ", TRAIN_DATASET_PATH)

input_transforms = transforms.Compose([
    transforms.ToTensor()
])
train_dataset = LabelDataset(
    images_dir=TRAIN_DATASET_PATH,
    transform=input_transforms
)

train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=1,
    pin_memory=0
)

print("Number of instances: ", len(train_loader))

Dataset path:  /home/vhondru/istoian/BMAD/dataset/BraTS2021_slice/train
Number of instances:  7500


In [4]:
count_pixels = 0
mean_ = torch.empty(3)
for images, _, _ in tqdm.tqdm(train_loader):
    crt_sum = torch.sum(images, dim=[0, 2, 3])
    mean_ = mean_ * count_pixels + crt_sum
    b, c, h, w = images.shape
    count_pixels += (b * h * w)
    mean_ /= count_pixels

print("Mean: ", mean_)

mean_ = torch.squeeze(mean_)
mean_ = torch.unsqueeze(mean_, 0)
mean_ = torch.unsqueeze(mean_, -1)
mean_ = torch.unsqueeze(mean_, -1)
print("Mean shape -> ", mean_.shape)

std_ = torch.empty(3)
count_pixels = 0
for images, _, _ in tqdm.tqdm(train_loader):
    crt_sum_sq = torch.sum(torch.pow(images - mean_, 2) , dim=[0, 2, 3])
    std_ = std_ * count_pixels + crt_sum_sq
    b, c, h, w = images.shape
    count_pixels += (b * h * w)
    std_ /= count_pixels

print("Standard deviation: ", std_)


100%|██████████| 7500/7500 [00:15<00:00, 492.21it/s]


Mean:  tensor([0.1129, 0.1240, 0.1445])
Mean shape ->  torch.Size([1, 3, 1, 1])


100%|██████████| 7500/7500 [00:16<00:00, 468.71it/s]

Standard deviation:  tensor([0.0392, 0.0495, 0.0612])


In [ ]:
# Mean:  tensor([0.1129, 0.1240, 0.1445])
# Standard deviation:  tensor([0.0392, 0.0495, 0.0612])